Ten skrypt ładuje surowe dane wejściowe do chmury Databricks

In [0]:
from threading import Thread

connection = spark.read.format("jdbc") \
    .option("url", "jdbc:mysql://***REMOVED***:3306/wheelie") \
    .option("user", "***REMOVED***") \
    .option("password", "***REMOVED***")

tables = (connection
    .option("query", "select table_name from information_schema.tables where table_schema = 'wheelie' and table_type <> 'view'")
    .load()
)

table_names = [r.table_name for r in tables.collect()]

tasks = []

for name in table_names:
    task = Thread(target = lambda: connection.option('query', f"select * from {name}").load().write.format("delta").mode("overwrite").saveAsTable(f"landing.{name}"))
    tasks.append(task)
    task.start()

for task in tasks:
    task.join()

kod po polaczeniu z baza, wykonuje zapytanie do meta danych w celu wyszukania wszyskich tabel w bazie.   
Nastepnie wielowatkowo nadpisuje obecne dane w tabelach w schema landing, na koniec synchronizuje wątki by miec pewność, że wszystkie się wykonały przed kolejnym krokiem